In [145]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [146]:
%cd /content

/content


In [147]:
!git clone https://github.com/highquanglity/LEAF-YOLO.git

fatal: destination path 'LEAF-YOLO' already exists and is not an empty directory.


In [148]:
%cd /content/LEAF-YOLO

import sys
import os
sys.path.insert(0, os.getcwd())

import torch

ckpt = torch.load(
    '/content/LEAF-YOLO/cfg/LEAF-YOLO/leaf-sizes/weights/best.pt',
    map_location='cuda',
    weights_only=False
)

print(ckpt.keys())

/content/LEAF-YOLO
dict_keys(['epoch', 'best_fitness', 'training_results', 'model', 'ema', 'updates', 'optimizer', 'wandb_id'])


In [149]:
pip install ultralytics

# Model Initialization

In [150]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cv2

In [151]:
model = ckpt['model'].float().cuda()

# Testing Detection

In [152]:
from utils.general import non_max_suppression, scale_coords
from utils.datasets import letterbox

In [153]:
img = cv2.imread("court.jpg")

#preprocessing image
test_img = letterbox(img, 640, stride=32)[0]
test_img = cv2.cvtColor(test_img, cv2.COLOR_BGR2RGB).transpose(2,0,1)
test_img = np.ascontiguousarray(test_img)
test_img = torch.from_numpy(test_img).float()/255.0
test_img = test_img.unsqueeze(0).cuda()

print(test_img.shape)



AttributeError: 'NoneType' object has no attribute 'shape'

In [154]:
##inference
model.eval()

with torch.no_grad():
  prediction = model(test_img)[0]

filtered_predictions = non_max_suppression(
    prediction,
    conf_thres=0.25,
    iou_thres=0.45
)

NameError: name 'test_img' is not defined

In [ ]:
for det in filtered_predictions:

  if len(det):
    det[:,:4] = scale_coords(test_img.shape[2:],det[:, :4], img.shape[:2]).round()

    for i in range(len(det)):
      x1 = int(det[i][0])
      y1 = int(det[i][1])
      x2 = int(det[i][2])
      y2 = int(det[i][3])

      conf = float(det[i][4])

      cls = int(det[i][5])

      if cls != 0:
        continue

      cv2.rectangle(img, (x1,y1), (x2,y2), (0,255,0), 2)

      label = f"Class {cls}: {conf:.2f}"

      cv2.putText(img, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2 )

cv2.imwrite("result.jpg", img)

# Detection for all Frames

In [155]:
frame_dir = "/content/drive/MyDrive/Data/VisDrone2019-MOT-val/VisDrone2019-MOT-val/sequences/uav0000086_00000_v"

In [156]:
import cv2
import os
import time

frame_files = sorted(os.listdir(frame_dir))

frames = []
run = 1
for file in frame_files:
  run+=1
  print(run)
  frame = cv2.imread(os.path.join(frame_dir, file))
  frames.append(frame)

detections = []

start_det = time.time()
for frame in frames:
  test = letterbox(frame, 640, stride=32)[0]
  test = cv2.cvtColor(test, cv2.COLOR_BGR2RGB).transpose(2,0,1)
  test = np.ascontiguousarray(test)
  test = torch.from_numpy(test).float()/255
  test = test.unsqueeze(0).cuda()

  model.eval()

  with torch.no_grad():
    prediction = model(test)[0]

  filtered_predictions = non_max_suppression(
    prediction,
    conf_thres=0.15,
    iou_thres=0.70
  )

  det = filtered_predictions[0]

  if det is not None and len(det):
    det[:,:4] = scale_coords(test.shape[2:],det[:, :4], frame.shape).round()

    det = det.cpu().numpy()

  else:
    det = np.empty((0, 6))

  detections.append(det)

end_det = time.time()


2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276
277
27

# Tracking

In [157]:
!git clone https://github.com/ifzhang/ByteTrack.git

fatal: destination path 'ByteTrack' already exists and is not an empty directory.


In [158]:
import sys

sys.path.append('/content/LEAF-YOLO')

In [159]:
!pip install lap cython_bbox

In [160]:
import os

print(os.listdir('/content'))

['.config', 'LEAF-YOLO', 'drive', 'sample_data']


In [161]:
!find /content/LEAF-YOLO -name "byte_tracker.py"

/content/LEAF-YOLO/ByteTrack/tutorials/cstrack/byte_tracker.py
/content/LEAF-YOLO/ByteTrack/tutorials/jde/byte_tracker.py
/content/LEAF-YOLO/ByteTrack/tutorials/centertrack/byte_tracker.py
/content/LEAF-YOLO/ByteTrack/tutorials/motr/byte_tracker.py
/content/LEAF-YOLO/ByteTrack/tutorials/transtrack/mot_online/byte_tracker.py
/content/LEAF-YOLO/ByteTrack/tutorials/trades/byte_tracker.py
/content/LEAF-YOLO/ByteTrack/tutorials/ctracker/byte_tracker.py
/content/LEAF-YOLO/ByteTrack/tutorials/fairmot/byte_tracker.py
/content/LEAF-YOLO/ByteTrack/tutorials/qdtrack/byte_tracker.py
/content/LEAF-YOLO/ByteTrack/yolox/tracker/byte_tracker.py


In [162]:
import sys

sys.path.append('/content/LEAF-YOLO/ByteTrack')

In [163]:
!pip install loguru

In [164]:
import numpy as np

np.float = float

In [165]:
pip install thop

In [166]:
from yolox.tracker.byte_tracker import BYTETracker

In [167]:
class TrackerArgs:

    track_thresh = 0.50
    track_buffer = 80
    match_thresh = 0.80
    mot20 = False

tracker = BYTETracker(
    TrackerArgs(),
    frame_rate=30
)

In [168]:
## Camera Compensation
def calculate_gmc(prev_gray, curr_gray):
    if prev_gray is None or curr_gray is None:
        return 0.0, 0.0
    prev_pts = cv2.goodFeaturesToTrack(prev_gray, maxCorners=150, qualityLevel=0.01, minDistance=20, blockSize=3)
    if prev_pts is None:
        return 0.0, 0.0
    curr_pts, status, _ = cv2.calcOpticalFlowPyrLK(prev_gray, curr_gray, prev_pts, None)
    good_prev = prev_pts[status == 1]
    good_curr = curr_pts[status == 1]
    if len(good_prev) < 4:
        return 0.0, 0.0
    matrix, _ = cv2.estimateAffinePartial2D(good_prev, good_curr)
    if matrix is None:
        return 0.0, 0.0
    return matrix[0, 2], matrix[1, 2]

In [169]:
height, width = frames[0].shape[:2]

video = cv2.VideoWriter(
    "output.mp4",
    cv2.VideoWriter_fourcc(*'mp4v'),
    30,
    (width, height)
)

prev_gray = None
my_results = []

start_track = time.time()
for frame, det in zip(frames, detections):
  curr_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

  dx, dy = 0.0, 0.0

  person_dets = det[det[:,5]==0]

  tracker_input = person_dets[:,:5].copy()

  if len(tracker_input) > 0:
      tracker_input[:, 0] -= dx
      tracker_input[:, 1] -= dy
      tracker_input[:, 2] -= dx
      tracker_input[:, 3] -= dy

  online_targets = tracker.update(tracker_input, img_info= frame.shape, img_size = frame.shape)
  frame_tracks = []

  for track in online_targets:
    track_id = track.track_id
    x1, y1, x2, y2 = map(int, track.tlbr)

    x1 = max(0, min(int(x1 + dx), width))
    y1 = max(0, min(int(y1 + dy), height))
    x2 = max(0, min(int(x2 + dx), width))
    y2 = max(0, min(int(y2 + dy), height))

    cv2.rectangle(frame, (x1, y1), (x2,y2), (0,255,0), 2)
    cv2.putText(frame, str(track_id), (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2 )

    frame_tracks.append({"id": track_id, "box": [x1, y1, x2 - x1, y2 - y1]})

  my_results.append(frame_tracks)
  prev_gray = curr_gray
  video.write(frame)

video.release()
end_track = time.time()

In [170]:
# 5. Calculate final FPS metrics
num_frames = len(frames)
total_det_time = end_det - start_det
total_track_time = end_track - start_track
total_pipeline_time = total_det_time + total_track_time

print("\n" + "="*50)
print("PERFORMANCE METRICS")
print("="*50)
print(f"Total Frames Processed: {num_frames}")
print(f"Detection Loop Time:    {total_det_time:.2f} seconds ({(num_frames / total_det_time):.2f} FPS)")
print(f"Tracking Loop Time:     {total_track_time:.2f} seconds ({(num_frames / total_track_time):.2f} FPS)")
print("-"*50)
print(f"TOTAL PIPELINE TIME:    {total_pipeline_time:.2f} seconds")
print(f"TOTAL PIPELINE FPS:     {(num_frames / total_pipeline_time):.2f} FPS")
print("="*50)


PERFORMANCE METRICS
Total Frames Processed: 464
Detection Loop Time:    18.86 seconds (24.60 FPS)
Tracking Loop Time:     16.58 seconds (27.98 FPS)
--------------------------------------------------
TOTAL PIPELINE TIME:    35.44 seconds
TOTAL PIPELINE FPS:     13.09 FPS


In [ ]:
pip install motmetrics

In [171]:
import motmetrics as mm
import numpy as np

gt_file_path = "/content/drive/MyDrive/Data/VisDrone2019-MOT-val/VisDrone2019-MOT-val/annotations/uav0000086_00000_v.txt"
if not hasattr(np, 'asfarray'):
    np.asfarray = lambda x: np.asarray(x, dtype=float)

# 1. Parse the Ground Truth
ground_truth_dict = {}
gt_data = np.loadtxt(gt_file_path, delimiter=',')
for row in gt_data:
    if len(row) >= 8 and int(row[7]) not in [1, 2]: # Only look at persons
        continue
    frame_idx = int(row[0]) - 1
    if frame_idx < len(frame_files):
        if frame_idx not in ground_truth_dict:
            ground_truth_dict[frame_idx] = {"ids": [], "boxes": []}
        ground_truth_dict[frame_idx]["ids"].append(int(row[1]))
        ground_truth_dict[frame_idx]["boxes"].append([row[2], row[3], row[4], row[5]])

# 2. Compare Ground Truth to Your Saved Results
accumulator = mm.MOTAccumulator(auto_id=True)

for frame_idx, tracking_data in enumerate(my_results):
    # tracking numbers
    det_ids = [t["id"] for t in tracking_data]
    det_boxes = [t["box"] for t in tracking_data]

    # Ground truth numbers
    if frame_idx in ground_truth_dict:
        gt_ids = ground_truth_dict[frame_idx]["ids"]
        gt_boxes = ground_truth_dict[frame_idx]["boxes"]
    else:
        gt_ids, gt_boxes = [], []

    # Calculate overlap (IoU)
    distance_matrix = mm.distances.iou_matrix(gt_boxes, det_boxes, max_iou=0.5)
    accumulator.update(gt_ids, det_ids, distance_matrix)

# 3. final report
mh = mm.metrics.create()
summary = mh.compute(
    accumulator,
    metrics=['num_frames', 'mota', 'idf1', 'num_switches', 'precision', 'recall'],
    name='VisDrone_Final_Evaluation'
)
print("\n" + "="*50)
print(mm.io.render_summary(summary, formatters=mh.formatters, namemap=mm.io.motchallenge_metric_names))
print("="*50)


                          num_frames  MOTA  IDF1 IDs  Prcn  Rcll
VisDrone_Final_Evaluation        464 44.5% 62.4%  27 80.8% 58.5%


In [ ]:
# height, width = frames[0].shape[:2]

# video = cv2.VideoWriter(
#     "output.mp4",
#     cv2.VideoWriter_fourcc(*'mp4v'),
#     30,
#     (width, height)
# )


# for frame, det in zip(frames, detections):
#   person_dets = det[det[:,5]==0]

#   tracker_input = person_dets[:,:5]

#   online_targets = tracker.update(tracker_input, img_info= frame.shape, img_size = frame.shape)

#   for track in online_targets:
#     track_id = track.track_id
#     x1, y1, x2, y2 = map(int, track.tlbr)

#     cv2.rectangle(frame, (x1, y1), (x2,y2), (0,255,0), 2)
#     cv2.putText(frame, str(track_id), (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2 )

#   video.write(frame)

# video.release()

# Batch Wise detection and then tracking

In [193]:
frame_files = sorted(os.listdir(frame_dir))

frames = []

for file in frame_files:
    frame = cv2.imread(os.path.join(frame_dir, file))
    frames.append(frame)

In [194]:
batch_size = 1 ## for live detection and tracking batch size = 1

In [195]:
detections = []

model.eval()

start_det = time.time()

for i in range(0, len(frames), batch_size):

    batch_frames = frames[i:i+batch_size]

    batch_tensor = []

    for frame in batch_frames:

        test = letterbox(frame, 640, stride=32)[0]
        test = cv2.cvtColor(test, cv2.COLOR_BGR2RGB).transpose(2,0,1)
        test = np.ascontiguousarray(test)

        test = torch.from_numpy(test).float() / 255

        batch_tensor.append(test)

    batch_tensor = torch.stack(batch_tensor).cuda()

    with torch.no_grad():
        predictions = model(batch_tensor)[0]

    filtered_predictions = non_max_suppression(
        predictions,
        conf_thres=0.20,
        iou_thres=0.70
    )

    for frame, det in zip(batch_frames, filtered_predictions):

        if det is not None and len(det):

            det[:, :4] = scale_coords(
                batch_tensor.shape[2:],
                det[:, :4],
                frame.shape
            ).round()

            det = det.cpu().numpy()

        else:

            det = np.empty((0, 6))

        detections.append(det)

end_det = time.time()

In [196]:
print("PERFORMANCE METRICS")
print(f"Total Frames Processed: {len(frames)}")
print(f"Detection Loop Time:    {end_det-start_det:.2f} seconds ({len(frames)/(end_det-start_det):.2f} FPS)")
print(f"Tracking Loop Time:     {end_track-start_track:.2f} seconds ({len(frames)/(end_track-start_track):.2f} FPS)")

total_time = (end_det-start_det) + (end_track-start_track)

print(f"TOTAL PIPELINE TIME:    {total_time:.2f} seconds")
print(f"TOTAL PIPELINE FPS:     {len(frames)/total_time:.2f} FPS")

PERFORMANCE METRICS
Total Frames Processed: 464
Detection Loop Time:    17.34 seconds (26.76 FPS)
Tracking Loop Time:     16.58 seconds (27.98 FPS)
TOTAL PIPELINE TIME:    33.92 seconds
TOTAL PIPELINE FPS:     13.68 FPS
